In [0]:
%sql
MERGE INTO catalog_ventas.gold.dim_producto AS tgt
USING (
    SELECT articulo, descrip, categoria, hash_producto
    FROM (
        SELECT
            articulo,
            TRIM(descrip) AS descrip,
            TRIM(categoria) AS categoria,
            md5(concat_ws('||', TRIM(descrip), TRIM(categoria))) AS hash_producto,
            ROW_NUMBER() OVER (
                PARTITION BY articulo
                ORDER BY _processing_timestamp DESC
            ) AS rn
        FROM catalog_ventas.silver.ventas_clean
        WHERE articulo IS NOT NULL
    )
    WHERE rn = 1
) AS src

ON tgt.articulo = src.articulo

WHEN MATCHED
AND COALESCE(tgt.hash_producto,'') <> COALESCE(src.hash_producto,'')
THEN UPDATE SET
    tgt.descrip = src.descrip,
    tgt.categoria = src.categoria,
    tgt.fecha_fin=CURRENT_DATE(),
    tgt.es_actual= FALSE,
    tgt._updated_at = CURRENT_TIMESTAMP()
    
WHEN NOT MATCHED THEN
INSERT (
    articulo,
    descrip,
    categoria,
    hash_producto,
    fecha_inicio,
    fecha_fin,
    es_actual,
    _created_at
)
VALUES (
    src.articulo,
    src.descrip,
    src.categoria,
    src.hash_producto,
    CURRENT_DATE(),
    '9999-01-01',
    TRUE,
    CURRENT_TIMESTAMP()
);

In [0]:
%sql
-- INSERTA REGISTROS QUE YA FUERON CERRADOS
INSERT INTO catalog_ventas.gold.dim_producto (
  articulo,
  descrip,
  categoria,
  hash_producto,
  fecha_inicio,
  fecha_fin,
  es_actual
)
SELECT
  src.articulo,
  src.descrip,
  src.categoria,
  src.hash_producto,
  CURRENT_DATE() AS fecha_inicio,
  '9999-01-01' AS fecha_fin,
  TRUE AS es_actual
FROM (
  SELECT
      articulo,
      TRIM(descrip) AS descrip,
      TRIM(categoria) AS categoria,
      md5(concat_ws('||',TRIM(descrip), TRIM(categoria))) AS hash_producto
  FROM catalog_ventas.silver.ventas_clean
  WHERE articulo IS NOT NULL
) src
INNER JOIN catalog_ventas.gold.dim_producto tgt
  ON tgt.articulo = src.articulo
  AND tgt.es_actual = FALSE
  AND tgt.fecha_fin = CURRENT_DATE() - INTERVAL 1 DAY
WHERE NOT EXISTS (
  SELECT 1
  FROM catalog_ventas.gold.dim_producto d
  WHERE d.articulo = src.articulo
    AND d.es_actual = TRUE
    AND d.fecha_inicio = d.fecha_inicio
);